# 🎯 Live Demo — Uncertainty-Aware BIST 100 Forecaster

**What this does in ~2 minutes:**
1. Loads the trained Transformer model
2. Takes the **last 30 days** of real BIST 100 data
3. Runs **50 Monte Carlo passes** to get prediction + uncertainty
4. Shows: *Direction forecast + Confidence score + Current market regime*

Upload `financial_data.csv` when prompted.

## Step 1: Install & Import

In [ ]:
!pip install torch pandas numpy scikit-learn hmmlearn matplotlib --quiet

import math, random, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from hmmlearn.hmm import GaussianHMM
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Ready — device: {device}')

## Step 2: Upload Data

In [ ]:
from google.colab import files
uploaded = files.upload()  # upload financial_data.csv

FEATURES = ['BIST100_Close', 'SP500_Close', 'CPIAUCSL', 'FEDFUNDS', 'INDPRO']
df = pd.read_csv(list(uploaded.keys())[0], index_col=0, parse_dates=True).sort_index()
df = df[FEATURES].dropna()

print(f'✅ Loaded {len(df)} days  |  {df.index.min().date()} → {df.index.max().date()}')
print(f'   Latest date in dataset: {df.index.max().date()}')
df.tail(3)

## Step 3: Define & Train Model (quick retrain on full data)

In [ ]:
# ── Model architecture (same as full training) ──────────────────────
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1), :])

class MCDropout(nn.Dropout):
    def forward(self, x):
        return nn.functional.dropout(x, self.p, training=True)

class UncertaintyTransformer(nn.Module):
    def __init__(self, input_dim=5, d_model=64, n_heads=4, n_layers=2, d_ff=128, dropout=0.1, mc_p=0.2):
        super().__init__()
        self.proj = nn.Linear(input_dim, d_model)
        self.pe   = PositionalEncoding(d_model, dropout=dropout)
        enc = nn.TransformerEncoderLayer(d_model, n_heads, d_ff, dropout, batch_first=True, activation='relu')
        self.encoder = nn.TransformerEncoder(enc, num_layers=n_layers)
        self.head = nn.Sequential(
            nn.Linear(d_model, 64), nn.ReLU(), MCDropout(mc_p),
            nn.Linear(64, 32),      nn.ReLU(), MCDropout(mc_p),
        )
        self.mu = nn.Linear(32, 1)
    def forward(self, x):
        x = self.pe(self.proj(x))
        x = self.encoder(x)[:, -1, :]
        return self.mu(self.head(x)).squeeze(-1)

# ── Quick training on TRAIN set (2010-2022) ─────────────────────────
LOOKBACK = 30
TRAIN_END = '2022-12-31'

df['target'] = np.log(df['BIST100_Close'].shift(-1) / df['BIST100_Close'])
df_clean = df.dropna()

train_df = df_clean.loc[:TRAIN_END]

scaler = StandardScaler().fit(train_df[FEATURES].values)
X_all  = scaler.transform(df_clean[FEATURES].values).astype(np.float32)
y_all  = df_clean['target'].values.astype(np.float32)

# Build windows for training
n_train = len(train_df)
X_tr, y_tr = [], []
for i in range(LOOKBACK-1, n_train):
    X_tr.append(X_all[i-LOOKBACK+1:i+1])
    y_tr.append(y_all[i])
X_tr = torch.from_numpy(np.stack(X_tr))
y_tr = torch.from_numpy(np.array(y_tr))

from torch.utils.data import TensorDataset, DataLoader
loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=64, shuffle=True)

model = UncertaintyTransformer().to(device)
opt   = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
crit  = nn.MSELoss()

print('Training model on 2010–2022 data...')
for epoch in range(1, 18):  # 17 epochs (same as full run)
    model.train()
    losses = []
    for Xb, yb in loader:
        Xb, yb = Xb.to(device), yb.to(device)
        opt.zero_grad()
        loss = crit(model(Xb), yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        losses.append(loss.item())
    if epoch % 5 == 0 or epoch == 17:
        print(f'  Epoch {epoch:2d} | Train MSE: {np.mean(losses):.6f}')

print('✅ Model trained!')

## Step 4: 🔮 LIVE FORECAST — Last 30 Days → Tomorrow's Prediction

In [ ]:
N_MC = 50

# Take the LAST 30 days of available data
last_window_raw = df_clean[FEATURES].values[-LOOKBACK:]
last_window     = scaler.transform(last_window_raw).astype(np.float32)
X_demo          = torch.from_numpy(last_window).unsqueeze(0).to(device)  # (1, 30, 5)

# Run 50 Monte Carlo passes
model.eval()  # MCDropout stays ON
with torch.no_grad():
    preds = [model(X_demo).item() for _ in range(N_MC)]

preds  = np.array(preds)
mu     = preds.mean()
sigma  = preds.std()
sigma2 = preds.var()

direction = 'UP ▲' if mu > 0 else 'DOWN ▼'
dir_color = '\033[92m' if mu > 0 else '\033[91m'  # green / red
reset     = '\033[0m'

# Confidence level based on sigma relative to test period average (0.0035)
avg_sigma = 0.0035
if sigma < avg_sigma * 0.8:
    conf_label = 'HIGH confidence'
    conf_note  = 'Model has seen similar patterns — trust this more'
elif sigma < avg_sigma * 1.3:
    conf_label = 'MODERATE confidence'
    conf_note  = 'Normal uncertainty level — trade with standard caution'
else:
    conf_label = 'LOW confidence'
    conf_note  = 'Unusual market conditions — reduce position size'

pred_date = df_clean.index[-1].date()

print('=' * 58)
print('   UNCERTAINTY-AWARE BIST 100 FORECAST')
print('=' * 58)
print(f'  Input window : {df_clean.index[-LOOKBACK].date()} → {pred_date}')
print(f'  Predicting   : next trading day after {pred_date}')
print(f'  MC passes    : N = {N_MC}')
print('-' * 58)
print(f'  Direction    : {dir_color}{direction}{reset}')
print(f'  μ (log ret)  : {mu:+.6f}')
print(f'  σ (uncert.)  : {sigma:.6f}')
print(f'  Confidence   : {conf_label}')
print(f'  Note         : {conf_note}')
print('=' * 58)

## Step 5: 📊 Visualise the 50 MC Predictions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left — distribution of 50 predictions
ax = axes[0]
color = '#06D6A0' if mu > 0 else '#EF476F'
ax.hist(preds, bins=15, color=color, edgecolor='white', alpha=0.85)
ax.axvline(mu, color='#1E2761', linewidth=2.5, label=f'μ = {mu:+.5f}')
ax.axvline(mu - 2*sigma, color='gray', linewidth=1.2, linestyle='--', alpha=0.7)
ax.axvline(mu + 2*sigma, color='gray', linewidth=1.2, linestyle='--', alpha=0.7, label='±2σ')
ax.set_xlabel('Predicted log return', fontsize=12)
ax.set_ylabel('Count (out of 50 passes)', fontsize=12)
ax.set_title(f'Distribution of {N_MC} MC Predictions\nDirection: {direction}', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)

# Right — last 60 days actual returns + forecast
ax2 = axes[1]
recent = df_clean['target'].iloc[-60:]
ax2.plot(range(len(recent)), recent.values, color='#3A86FF', linewidth=1.2, alpha=0.7, label='Actual returns (last 60 days)')
ax2.axhline(0, color='gray', linewidth=0.8, linestyle='--')

# Forecast bar
ax2.bar(len(recent), mu, color=color, alpha=0.9, width=0.6, label=f'Forecast μ = {mu:+.5f}')
ax2.errorbar(len(recent), mu, yerr=2*sigma, fmt='none', color='#1E2761', capsize=6, linewidth=2)

ax2.set_xlabel('Days (last 60 + forecast)', fontsize=12)
ax2.set_ylabel('Log return', fontsize=12)
ax2.set_title(f'Recent Returns + Tomorrow\'s Forecast\nσ = {sigma:.5f}  ({conf_label})', fontsize=13, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('demo_forecast.png', dpi=130, bbox_inches='tight')
plt.show()
print('✅ Saved demo_forecast.png')

## Step 6: 🗺️ Current Market Regime

In [ ]:
# Build regime features on full dataset
df_regime = df_clean[['BIST100_Close']].copy()
df_regime['log_return']  = np.log(df_regime['BIST100_Close'] / df_regime['BIST100_Close'].shift(1))
df_regime['rolling_vol'] = df_regime['log_return'].rolling(20).std()
df_regime = df_regime.dropna()

X_hmm = df_regime[['log_return', 'rolling_vol']].values

hmm = GaussianHMM(n_components=3, covariance_type='full', n_iter=200, random_state=42)
hmm.fit(X_hmm)
states = hmm.predict(X_hmm)
posteriors = hmm.predict_proba(X_hmm)

# Label states by mean return
order = np.argsort(hmm.means_[:, 0])
label_map = {order[0]: 'Bear', order[1]: 'Sideways', order[2]: 'Bull'}
color_map  = {'Bear': '#EF476F', 'Sideways': '#64748B', 'Bull': '#06D6A0'}

current_state      = states[-1]
current_regime     = label_map[current_state]
current_posteriors = posteriors[-1]
regime_date        = df_regime.index[-1].date()

print('=' * 58)
print('   CURRENT MARKET REGIME')
print('=' * 58)
print(f'  As of       : {regime_date}')
print(f'  Regime      : {current_regime}')
print(f'  P(Bull)     : {current_posteriors[order[2]]:.1%}')
print(f'  P(Sideways) : {current_posteriors[order[1]]:.1%}')
print(f'  P(Bear)     : {current_posteriors[order[0]]:.1%}')
print('=' * 58)

# Combined summary
print()
print('━' * 58)
print('   COMBINED FORECAST SUMMARY')
print('━' * 58)
print(f'  Market regime  : {current_regime}')
print(f'  Direction      : {direction}')
print(f'  Confidence     : {conf_label}')
print(f'  Recommendation : ', end='')

if current_regime == 'Bear' or conf_label == 'LOW confidence':
    print('⚠️  Be cautious — reduce position size')
elif current_regime == 'Bull' and conf_label == 'HIGH confidence':
    print('✅  Conditions favorable — can trade with conviction')
else:
    print('ℹ️  Moderate conditions — standard risk management')
print('━' * 58)

## Step 7: 📈 Regime History Plot

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 7))

bist_aligned = df_clean['BIST100_Close'].reindex(df_regime.index)

# Top: BIST price with regime shading
ax = axes[0]
ax.plot(df_regime.index, bist_aligned, color='#1E2761', linewidth=0.8, alpha=0.8, label='BIST 100')
for i, idx in enumerate(df_regime.index):
    regime = label_map[states[i]]
    ax.axvspan(idx, idx, alpha=0.12, color=color_map[regime], linewidth=0)
legend_patches = [mpatches.Patch(color=color_map[r], label=r) for r in ['Bull','Sideways','Bear']]
ax.legend(handles=legend_patches + [plt.Line2D([0],[0],color='#1E2761',label='BIST 100')], fontsize=9)
ax.set_title('BIST 100 Price with HMM Market Regimes', fontsize=13, fontweight='bold')
ax.set_ylabel('Price (TRY)')
ax.grid(alpha=0.3)

# Bottom: Posterior probabilities
ax2 = axes[1]
for state_id, name in label_map.items():
    ax2.plot(df_regime.index, posteriors[:, state_id],
             color=color_map[name], linewidth=0.8, alpha=0.8, label=f'P({name})')
ax2.axvline(df_regime.index[-1], color='black', linewidth=1.5, linestyle='--', label='Today')
ax2.set_ylim(0, 1)
ax2.set_title('HMM Posterior Probabilities per Regime', fontsize=13, fontweight='bold')
ax2.set_ylabel('Probability')
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('demo_regimes.png', dpi=130, bbox_inches='tight')
plt.show()
print('✅ Saved demo_regimes.png')